# 12 — Per-acquisition Sentinel-1 and daily rainfall, 2022–2025

**Status: SOURCE WORKFLOW ONLY — not executed or validated.** Running this notebook requires Earth Engine access and a local project boundary. It exports actual results only after successful extraction; there are no invented per-scene results in the committed file.

Study question: can individual Sentinel-1 scenes distinguish the 2022, 2024 and 2025 high-June screening extents (with 2023 as a low-June comparator) when monthly medians cannot?

- Compare Sentinel-1 ascending orbit 41 and descending orbit 77 **separately**; never combine VV intensities as if geometry is equivalent.
- Per-scene fixed −19/−17/−15 dB VV thresholds, incidence-angle screen, 30 m focal median with ORIGINAL image support mask preserved.
- Report acquisition timestamps in UTC and Asia/Dhaka. CHIRPS daily image DATE LABELS are not assumed to equal Bangladesh midnight-to-midnight rainfall totals. Rainfall-date offsets of −1/0/+1 day are sensitivity alternatives, not corrections.
- Area is LOW-VV SCREENING, not validated water extent, depth, damage or flood severity. First/last observed detections do not resolve onset or recession between passes.
- Inputs: study-area boundary, original Notebook 09 scene inventory, Earth Engine Sentinel-1 GRD and CHIRPS daily. Outputs: CSVs, figures, QA report and optional separate per-orbit frequency maps.


## 12A — Configuration, boundary, Earth Engine and reproducibility

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import ee

start = Path(os.environ.get("HAKALUKI_ROOT", Path.cwd())).expanduser().resolve()
for candidate in [start, *start.parents]:
    if ((candidate/"data/boundary/hakaluki_boundary.gpkg").is_file()
        or (candidate/"Study area/Hakaluki_Haor.shp").is_file()):
        ROOT = candidate
        break
else:
    raise FileNotFoundError("Set HAKALUKI_ROOT to the research project or start Jupyter inside it")
sys.path.insert(0, str(ROOT))
from src.scene12_utils import normalize_daily_rain, attach_rain_to_scenes, check_scene_areas, describe_scene_support

BOUNDARY = ROOT/"data/boundary/hakaluki_boundary.gpkg"
if not BOUNDARY.is_file():
    BOUNDARY = ROOT/"Study area/Hakaluki_Haor.shp"
TABLES, FIGURES, CACHE = ROOT/"results/tables", ROOT/"results/figures", ROOT/"data/processed/scene12_cache"
for folder in (TABLES, FIGURES, CACHE):
    folder.mkdir(parents=True, exist_ok=True)

YEARS = (2022, 2023, 2024, 2025)
MONTHS = (5, 9)             # May through September inclusive
TRACKS = (("ASCENDING", 41), ("DESCENDING", 77))
THRESHOLDS_DB = (-19.0, -17.0, -15.0)
ANGLE_RANGE = (30.0, 45.0)
FILTER_RADIUS_M, SCALE_M = 30, 30
MIN_SCENE_COVERAGE = 0.90
FORCE_RECOMPUTE = False
DO_VV_VH_SAMPLES = True

haor = gpd.read_file(BOUNDARY).to_crs("EPSG:4326")
assert len(haor) and haor.geometry.notna().all() and haor.geometry.is_valid.all()
polygon = haor.geometry.union_all()
project = os.getenv("EE_PROJECT") or os.getenv("GOOGLE_CLOUD_PROJECT")
try:
    if project:
        ee.Initialize(project=project)
    else:
        ee.Initialize()
except Exception as err:
    raise RuntimeError("Initialize Earth Engine and set EE_PROJECT locally if required; do not publish credentials") from err
ROI = ee.Geometry(polygon.__geo_interface__)
roi_stats = ee.Image.pixelArea().rename("roi_m2").reduceRegion(
    reducer=ee.Reducer.sum(), geometry=ROI, crs="EPSG:32646",
    scale=SCALE_M, maxPixels=100_000_000, tileScale=4).getInfo()
REFERENCE_KM2 = float(roi_stats["roi_m2"])/1e6
assert 10 < REFERENCE_KM2 < 500

CONFIG = {"years": list(YEARS), "months": list(MONTHS), "tracks": [list(x) for x in TRACKS],
          "thresholds": list(THRESHOLDS_DB), "angle": list(ANGLE_RANGE),
          "radius_m": FILTER_RADIUS_M, "scale_m": SCALE_M,
          "roi_wkb_hash": hashlib.sha256(polygon.wkb).hexdigest(),
          "source": "COPERNICUS/S1_GRD", "version": "12.1"}
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12]
print("Project folder:", ROOT, "| EE reference area:", round(REFERENCE_KM2,3), "km²")
print("Configuration hash:", CONFIG_HASH, " | no new result until executed")
